# 03. Anomaly Transformer와 원인 추적 (XAI)

> **Day 01 — 제조 시계열 AI (4/5)**
> 다변량 센서 데이터에서 이상 구간을 탐지하고, Attention 가중치로
> 어떤 센서가 원인인지 역추적한다.
>
> **이 노트북이 회수하는 난제**: ① 해석 가능성 — "왜 고장이라고 판단했는가"

---

## 📖 이 노트북의 스토리라인

> **"이상한 순간은 바로 옆 시점하고만 대화합니다 — 과거에 닮은 순간이 없으니까요."**

```
문제를 본다        기존 방법의 벽        새 착상            하이라이트
펌프 실데이터  →  AutoEncoder 한계  →  Association  →  Attention으로 원인 센서 지목
(라벨이 없다)     (왜인지 말 못 한다)    Discrepancy      (그리고 그 지목을 채점한다)
```

**⚠️ 이 노트북은 데이터를 세 벌 씁니다. 각각 역할이 다릅니다.**

| 구간 | 데이터 | 왜 이걸 쓰는가 |
|---|---|---|
| §1 문제 정의 | **수처리장 펌프** (실데이터, 5개월·고장 7건) | 라벨이 얼마나 희소한지, 라벨 정의가 왜 함정인지는 **실제 설비**로 봐야 한다 |
| §3 AE의 한계 | **펌프 테스트베드** (실데이터, SKAB) | Over-generalization은 **고장 라벨이 충분한** 데이터에서만 관찰된다 |
| §2·§4~§10 | **압출기 8채널** (합성) | 이상 3유형과 **원인 센서 정답**이 있어야 XAI가 맞았는지 채점할 수 있다 |

> 실데이터에는 "어느 센서가 원인인가"의 정답이 없습니다.
> 그래서 **문제 인식은 실데이터로, 알고리즘 검증은 정답이 있는 합성 데이터로** 나눴습니다.

| | |
|---|---|
| **이 노트북의 역할** | **난제 ① 회수.** 탐지에서 끝내지 않고 "왜"에 답한다 |
| **앞에서 이어받는 것** | NB02에서 직접 만든 Attention·Encoder 구조를 그대로 재사용 |
| **다음으로 넘기는 것** | "정상 데이터가 충분하다"는 전제 → NB04가 그 전제를 깬다 |

---

> **실습 안내**
> `"""따라하기"""` 가 적힌 셀은 강사와 함께 직접 실행합니다. 주석을 보고 코드를 채워 주세요.
> `"""직접구현"""` 이 적힌 셀은 여러분이 직접 채워 봅니다. 정답은 노트북 맨 아래에 있습니다.
> 나머지 셀은 실행 결과를 확인하며 따라오시면 됩니다.
> 막히는 부분은 손을 들어 주세요.

## 목차
1. 이상탐지 문제 정의 — 실제 펌프 설비 5개월 기록
2. 제조 이상의 3유형 — Point · Contextual · Collective
3. AutoEncoder 접근과 그 한계 — Over-generalization
4. Anomaly Transformer의 착상 — Prior · Series · Association Discrepancy
5. Min-Max 전략으로 학습하기
6. 이상 점수와 임계값 설정
7. 불균형 평가 — Accuracy를 버리고 PR로
8. **[핵심] Attention 히트맵 XAI — 원인 센서 특정**
9. 운영 관점 — 알람 피로와 비용 비대칭
10. AE vs Anomaly Transformer 비교

In [ ]:
# 공통 준비 — Colab / 로컬 양쪽에서 동작
import os, random, warnings
import numpy as np
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"PyTorch {torch.__version__} | device: {DEVICE}")
except ImportError:
    DEVICE = "cpu"
    print("PyTorch 미설치 — 다음 셀에서 설치합니다.")

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False
print(f"Colab 환경: {IN_COLAB}")

In [ ]:
# ══ 실습 자료 위치 — 강사가 배포한 주소로 이 한 줄만 맞추면 됩니다 ══════════
DATA_REPO = "https://raw.githubusercontent.com/leejiyoon52/ai-course/main/day1"

# Google Drive로 배포받았다면, 위 줄 대신 아래 두 줄의 주석을 푸십시오.
# from google.colab import drive; drive.mount("/content/drive")
# DATA_REPO = "file:///content/drive/MyDrive/ai-course/day1"
# ═══════════════════════════════════════════════════════════════════════

import os, shutil, urllib.request
os.environ["MFG_DATA_BASE"] = DATA_REPO          # loaders.py가 이 값을 읽습니다

for f in ["mfg_datagen.py", "loaders.py"]:
    if os.path.exists(f):
        continue
    src = f"{DATA_REPO}/modules/{f}"
    try:
        if src.startswith("file://"):
            shutil.copy(src[len("file://"):], f)
        else:
            urllib.request.urlretrieve(src, f)
        print(f"다운로드 완료: {f}")
    except Exception:
        print(f"⚠️ {f} 를 받지 못했습니다 — 왼쪽 파일 탭에 직접 업로드해 주세요")

import mfg_datagen
import loaders
print(f"실습 모듈 준비 완료 — 자료 위치: {DATA_REPO}")

---
## 1. 이상탐지 문제 정의 — 예측과 무엇이 다른가

> 이상탐지는 숙련 작업자의 *"오늘 소리가 평소와 다른데?"* 를 코드로 옮기는 일입니다.

NB02의 RUL 예측과 결정적으로 다른 점이 있습니다.

| | RUL 예측 (NB02) | 이상탐지 (NB03) |
|---|---|---|
| 라벨 | 모든 시점에 있음 (고장까지 남은 사이클) | **거의 없음.** 고장은 몇 건뿐 |
| 학습 방식 | 지도학습 | **정상 데이터만으로 학습**(One-class) |
| 판단 | "얼마나 남았는가" | "지금 평소와 다른가" |

실제 설비 데이터로 이 전제를 확인합니다. **수처리장 펌프 1대의 5개월 기록**입니다.

In [ ]:
# 펌프 설비 실데이터 로드 — 2018년 5개월, 5분 간격
"""따라하기"""


In [ ]:
# 데이터 검진부터 — NB01에서 만든 습관 그대로
"""따라하기"""


In [ ]:
# 고장 7건이 신호에 어떻게 나타나는가
"""따라하기"""


### 라벨의 진실 — 무엇을 "이상"이라 부를 것인가

`BROKEN` 은 고장이 **일어난** 순간이고, `RECOVERING` 은 수리·재가동 중인 사후 구간입니다.
둘을 합쳐 "이상"이라 부르면 라벨이 6.6%로 늘어 다루기 편해 보입니다.
그런데 여기에 함정이 있습니다.

> **현장 노트**
> PdM PoC에서 "AUC 0.99 달성" 보고를 받으면 저는 라벨 정의부터 확인합니다.
> 멈춘 설비를 "이상"이라고 맞히는 모델은 AUC가 거의 1이 나옵니다. 당연합니다,
> 센서값이 전부 바닥이니까요. 하지만 그건 이미 현장 작업자가 알고 있는 사실입니다.
> **돈이 되는 것은 고장 나기 전에 아는 것**이고, 그 문제는 난이도가 완전히 다릅니다.
> 다음 두 셀에서 그 격차를 숫자로 보겠습니다.

In [ ]:
# 함정 실증 — 같은 데이터·같은 점수인데 라벨 정의만 바꿔 본다
"""따라하기"""


---
## 2. 제조 이상의 3유형

실제 펌프 데이터로 문제의 성격을 봤습니다. 이제 **알고리즘의 원리를 검증**하려면
정답이 필요합니다 — 어떤 시점이 이상인지, 그리고 **어느 센서가 원인인지**까지.
실데이터에는 그 정답(원인 센서 라벨)이 없으므로, 이후 실습은 **압출기 8채널
합성 데이터**로 진행합니다. 원인 라벨이 있어야 XAI가 맞았는지 채점할 수 있습니다.

| 유형 | 비유 | 특징 |
|---|---|---|
| **Point** | 갑작스런 굉음 | 한 시점이 튄다. 눈에 잘 띈다 |
| **Contextual** | 한여름의 히터 가동 | **값 자체는 정상 범위**인데 맥락상 이상하다 |
| **Collective** | 리듬 자체가 어긋난 구간 | 개별 값은 정상, 구간 전체의 패턴이 깨진다 |

In [ ]:
# 압출기 다변량 데이터 생성 — 이상 3종 + 원인 센서 정답 포함
"""따라하기"""


In [ ]:
# 3유형을 하나씩 눈으로 확인 — 원인 센서만 그린다
picks = {}
for start, L, kind, sensor in rc["segments"]:
    picks.setdefault(kind, (start, L, sensor))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, kind in zip(axes, ["point", "contextual", "collective"]):
    s0, L, sensor = picks[kind]
    lo, hi = max(0, s0 - 80), s0 + L + 80
    ax.plot(range(lo, hi), raw[sensor].values[lo:hi], lw=0.9)
    ax.axvspan(s0, s0 + L, color="red", alpha=0.2)
    ax.set_title(f"{kind}\n({sensor})")
plt.tight_layout(); plt.show()
print("Point는 튀어서 보이지만, Contextual은 값의 높이만 보면 정상 범위 안에 있습니다.")

In [ ]:
# Contextual 이상은 '이웃 센서와의 관계'가 깨진 것 — 두 센서를 겹쳐 본다
s0, L, sensor = picks["contextual"]
other = [c for c in SENS if c != sensor][0]
lo, hi = max(0, s0 - 120), s0 + L + 120

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(range(lo, hi), raw[sensor].values[lo:hi], label=f"{sensor} (root cause)")
ax.plot(range(lo, hi), raw[other].values[lo:hi], label=other, alpha=0.7)
ax.axvspan(s0, s0 + L, color="red", alpha=0.15)
ax.set_title("Contextual anomaly: value in range, relationship broken")
ax.legend(); plt.tight_layout(); plt.show()
print("붉은 구간에서 두 센서가 반대로 움직입니다. 단일 센서 임계값으로는 절대 잡히지 않습니다.")

---
## 3. AutoEncoder 접근과 그 한계

가장 널리 쓰이는 출발점은 **AutoEncoder(AE)** 입니다.

1. 정상 데이터만으로 "압축했다가 복원하는" 법을 배우게 합니다
2. 이상 데이터는 배운 적이 없으니 복원이 서툴 것이다 → **재구성 오차가 크면 이상**

논리는 깔끔합니다. 그런데 실제 설비 데이터로 돌려 보면 예상 밖의 일이 일어납니다.

§1의 펌프는 5개월에 고장이 7건뿐이라 검증에 쓸 이상이 부족했습니다.
그래서 이번에는 **같은 수순환 펌프 계열의 테스트베드 데이터**를 씁니다.
고장을 의도적으로 유도해 기록한 벤치마크라 라벨이 충분합니다.

In [ ]:
# 펌프 테스트베드 데이터 로드 — 정상 구간과 고장 유도 구간
bench_tr, bench_te, bench_y = loaders.load_anomaly()
BSENS = list(bench_tr.columns)
print(f"학습(정상 가동) {bench_tr.shape} | 평가 {bench_te.shape}")
print(f"평가 구간 이상 비율: {100 * bench_y.mean():.2f}%")

bsc = StandardScaler().fit(bench_tr)
btr = bsc.transform(bench_tr).astype(np.float32)
bte = bsc.transform(bench_te).astype(np.float32)

In [ ]:
# AutoEncoder 정의 — 윈도우를 통째로 압축했다 편다
"""따라하기"""


In [ ]:
# AE 학습 — epoch마다 '재구성 손실'과 '탐지 성능'을 같이 기록한다
# 학습 약 60초 소요 (T4 기준)
import time

B_tr = to_windows(btr, stride=WIN // 4)
B_te = to_windows(bte, stride=1)
b_center = bench_y.values[WIN // 2: WIN // 2 + len(B_te)]      # 윈도우 중앙 시점 라벨

torch.manual_seed(SEED)
ae_b = AutoEncoder(len(BSENS), WIN).to(DEVICE)
opt_b = torch.optim.Adam(ae_b.parameters(), lr=1e-3)
dl_b = DataLoader(TensorDataset(torch.tensor(B_tr)), batch_size=64, shuffle=True)
Bte_t = torch.tensor(B_te).to(DEVICE)

hist = []
t0 = time.time()
for ep in range(20):
    ae_b.train(); tot = 0.0
    for (xb,) in dl_b:
        xb = xb.to(DEVICE)
        opt_b.zero_grad(); loss = ((ae_b(xb) - xb) ** 2).mean()
        loss.backward(); opt_b.step(); tot += loss.item()
    ae_b.eval()
    with torch.no_grad():
        err_b = ((ae_b(Bte_t) - Bte_t) ** 2).mean(dim=(1, 2)).cpu().numpy()
    hist.append((ep + 1, tot / len(dl_b), roc_auc_score(b_center, err_b)))
    if ep % 4 == 0 or ep == 19:
        print(f"epoch {ep + 1:2d} | 재구성 손실 {hist[-1][1]:.4f} | 탐지 ROC-AUC {hist[-1][2]:.3f}")
print(f"학습 시간 {time.time() - t0:.0f}초")

In [ ]:
# ★ 두 곡선을 겹쳐 본다 — 손실이 내려가는 동안 탐지력은 어떻게 움직였는가
h = np.array(hist)
fig, ax1 = plt.subplots(figsize=(10, 3.5))
ax1.plot(h[:, 0], h[:, 1], color="tab:blue", marker="o", ms=3)
ax1.set_xlabel("epoch"); ax1.set_ylabel("reconstruction loss", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(h[:, 0], h[:, 2], color="tab:red", marker="s", ms=3)
ax2.set_ylabel("detection ROC-AUC", color="tab:red")
best_ep = int(h[np.argmax(h[:, 2]), 0])
ax1.axvline(best_ep, ls="--", color="gray")
ax1.set_title("Reconstruction loss (blue) vs detection ability (red)")
plt.tight_layout(); plt.show()

peak, final = h[:, 2].max(), h[-1, 2]
print(f"탐지 성능 정점: epoch {best_ep} (ROC-AUC {peak:.3f}) → 최종 epoch {int(h[-1, 0])} ({final:.3f})")
if final < peak - 0.02:
    print("\n▶ 손실은 끝까지 내려갔는데 탐지력은 꺾였습니다. 이것이 Over-generalization입니다.")
    print("  학습을 더 할수록 AE가 이상 파형까지 복원해 버려 오차 차이가 줄어든 것입니다.")
else:
    print("\n▶ 이 실행에서는 탐지력이 끝까지 유지됐습니다. 이상이 정상 분포에서 충분히 멀었기 때문입니다.")
    print("  Over-generalization은 이상이 정상과 가까울 때 나타납니다 — 다음 셀 설명을 참고하십시오.")
print("\n어느 쪽이든 결론은 같습니다: 손실 곡선은 탐지력을 대변하지 못합니다. 반드시 따로 재야 합니다.")

### *너무 유능한 복사기* — Over-generalization

AutoEncoder는 **"입력을 잘 복원하라"** 만 배웠지, **"이상은 복원하지 말라"** 는 배운 적이 없습니다.
학습이 길어질수록 처음 보는 파형까지 그럭저럭 복원해 버립니다.
불량품까지 똑같이 복사해 내는 복사기입니다.

**언제 심해지는가**

| 이상의 성격 | AE의 운명 |
|---|---|
| 정상 분포에서 **멀다** (설비 정지처럼 값이 바닥으로) | 오래 학습해도 잘 버팁니다 |
| 정상 분포와 **가깝다** (미세한 마모, 맥락상 이상) | 학습이 길어질수록 탐지력이 무너집니다 |

그래서 "며칠 더 돌렸더니 성능이 떨어졌다"는 보고를 받으면 버그보다 이 현상을 먼저 의심합니다.
대응은 하나입니다 — **재구성 손실이 아니라 탐지 지표로 조기 종료를 판단**하는 것입니다.

**그리고 더 중요한 한계**

이 노트북의 주제와 직결되는 문제가 남아 있습니다.
**재구성 오차는 "이상하다"고만 말할 뿐, "왜"를 말하지 않습니다.**
현장 엔지니어에게 건넬 근거가 점수 하나뿐입니다.
이제 탐지와 설명을 함께 주는 구조로 넘어갑니다.

In [ ]:
# 실습 데이터 전환 — 원인 센서 정답이 있는 압출기 데이터로
"""따라하기"""


In [ ]:
# 압출기 데이터로 AE 기준선 확보 — 두 규모로 학습해 둔다
"""따라하기"""


---
## 4. Anomaly Transformer의 착상

논문의 관찰은 한 문장입니다.

> **정상 구간은 멀리 있는 시점과도 대화합니다.** 주기적으로 반복되니까요.
> **이상 구간은 바로 옆 시점하고만 대화합니다.** 과거에 닮은 순간이 없기 때문입니다.

이 "대화 상대의 범위 차이"를 두 개의 분포로 만들어 비교합니다.

| 이름 | 정체 | 의미 |
|---|---|---|
| **Prior-Association** $P$ | 학습되는 가우시안 분포 | "인접 시점에 집중한다"는 가정을 표현 |
| **Series-Association** $S$ | 실제 학습된 Attention | 모델이 진짜로 어디를 보는가 |
| **Association Discrepancy** | 두 분포의 거리 | 이 값이 **작으면 이상** |

$$\text{AssDis}(t) = \text{KL}\big(P_t \,\|\, S_t\big) + \text{KL}\big(S_t \,\|\, P_t\big)$$

정상 시점은 $S$가 멀리까지 퍼져 $P$(인접 집중)와 크게 다릅니다 → AssDis 큼.
이상 시점은 $S$도 옆만 보므로 $P$와 닮습니다 → AssDis 작음.

In [ ]:
# Prior-Association 구현 — 거리에 따라 감쇠하는 가우시안
"""따라하기"""


In [ ]:
# Association Discrepancy — 두 분포의 대칭 KL 거리
"""직접구현"""
def assoc_discrepancy(P, S, eps=1e-8):
    """P, S: (B, h, L, L) → 시점별 불일치도 (B, L)"""
    # TODO: KL(P||S) 와 KL(S||P) 를 각각 계산 (마지막 축으로 sum)
    #       두 값을 더한 뒤 head 축(dim=1)으로 평균
    #       반환 shape: (B, L)
    raise NotImplementedError('직접 채워 보세요')


# 검증: 옆만 보는 Attention은 prior와 닮아 AssDis가 작아야 한다
P_demo = make_prior(40, 3.0)[None, None]
S_near = make_prior(40, 3.0)[None, None]                 # 이상 시점처럼 인접만
S_far = make_prior(40, 15.0)[None, None]                 # 정상 시점처럼 멀리까지
print(f"이상처럼 옆만 볼 때 AssDis: {assoc_discrepancy(P_demo, S_near).mean():.4f}")
print(f"정상처럼 멀리 볼 때 AssDis: {assoc_discrepancy(P_demo, S_far).mean():.4f}")
print("→ 이상 시점의 AssDis가 더 작습니다. 이 성질이 탐지의 근거가 됩니다.")


In [ ]:
# Anomaly-Attention 조립 — Series와 Prior를 함께 내놓는 층
"""따라하기"""


In [ ]:
# Anomaly Transformer 조립 — NB02의 Encoder 구조를 그대로 재사용
"""따라하기"""


---
## 5. Min-Max 전략으로 학습하기

여기가 이 논문의 묘수입니다. 두 분포를 그냥 두면 모델이 편한 쪽으로 붙여 버려
차이가 사라집니다. 그래서 **밀고 당기는 두 단계**를 번갈아 적용합니다.

| 단계 | 목적 | 손실 |
|---|---|---|
| **Minimize** | Prior가 Series를 따라가게 (P를 현실에 맞춤) | `재구성 − k · AssDis(P, S.detach())` |
| **Maximize** | Series는 Prior에서 멀어지게 (정상 시점이 더 멀리 보게) | `재구성 + k · AssDis(P.detach(), S)` |

`detach()` 가 핵심입니다. 한 쪽을 고정한 채 다른 쪽만 움직여야 밀당이 성립합니다.
그 결과 **정상 시점의 AssDis는 커지고, 이상 시점은 옆밖에 볼 게 없어 작게 남습니다.**
두 집단의 간격이 벌어지는 것 — 그것이 이 학습의 목표입니다.

In [ ]:
# Min-Max 학습 — 한 배치에 두 번의 역전파
"""따라하기"""


In [ ]:
# 사전학습 체크포인트 로드 — 없으면 방금 학습한 데모 모델로 진행
"""따라하기"""


---
## 6. 이상 점수와 임계값 설정

두 신호를 결합해 최종 점수를 만듭니다.

$$\text{Score}(t) = \underbrace{\text{Softmax}\big(-\text{AssDis}\big)_t}_{\text{옆만 보는 시점일수록 큼}} \times \underbrace{\lVert x_t - \hat{x}_t \rVert^2}_{\text{재구성 오차}}$$

재구성 오차만으로는 부족하고(AE의 한계), AssDis만으로도 부족합니다.
**"닮은 과거가 없으면서 동시에 복원도 안 되는 시점"** 이 진짜 이상입니다.

> **구현 주의 — 정규화 범위**
> 위 softmax를 **윈도우 하나 안에서** 계산하면 안 됩니다.
> 이상이 한 건도 없는 깨끗한 윈도우에서도 softmax가 억지로 최댓값을 만들어
> 허위 알람이 됩니다. 두 신호 모두 **전체 구간 기준으로 표준화**한 뒤 더합니다.
> 논문 구현도 시퀀스 전체에 걸쳐 정규화합니다.

### 그리고 어디서 자를 것인가 — 임계값은 알람 민감도 다이얼입니다

> *알람 민감도 다이얼 — 돌릴수록 오탐과 미검출이 시소를 탑니다.*

순서를 뒤집는 것이 실무의 요령입니다. "성능이 가장 좋은 임계값"을 찾는 것이 아니라,
**하루에 감당할 수 있는 알람 건수를 먼저 정하고** 거기에 맞춰 자릅니다.
알람을 받는 사람이 정해진 인원이기 때문입니다.

---
## 7. 불균형 평가 — Accuracy를 버리고 PR로

NB00에서 예고한 장면입니다. 이상 비율 1% 남짓인 데이터에서
Accuracy는 성적표 역할을 하지 못합니다. 임계값을 정한 뒤 바로 이어서 확인합니다.

In [ ]:
# 이상 점수 산출 — AssDis와 재구성 오차를 전체 기준으로 표준화해 결합
"""직접구현"""
def anomaly_score(model, X, batch=64):
    """반환: score (N, L), sensor_err (N, L, C), assdis (N, L), series_attn (N, L, L)"""
    # TODO: 배치를 돌며 재구성 오차 (B,L,C) 와 AssDis (B,L) 를 모은다
    #       err 과 -assdis 를 각각 전체 기준으로 표준화한 뒤 더한다
    #       반환: score(N,L), sensor_err(N,L,C), assdis(N,L), attn(N,L,L)
    raise NotImplementedError('직접 채워 보세요')


score, sensor_err, assdis_te, attn = anomaly_score(model, X_te)
print(f"score {score.shape} | 센서별 오차 {sensor_err.shape} | attention {attn.shape}")
print(f"정상 시점 평균 AssDis {assdis_te.ravel()[y_flat == 0].mean():.4f} | "
      f"이상 시점 평균 AssDis {assdis_te.ravel()[y_flat == 1].mean():.4f}")


In [ ]:
# AssDis가 정말 보탬이 되었는가 — 두 신호를 따로, 그리고 함께 채점한다
err_only = sensor_err.mean(-1).ravel()
ad_only = -assdis_te.ravel()

parts = pd.DataFrame({
    "ROC-AUC": [roc_auc_score(y_flat, err_only), roc_auc_score(y_flat, ad_only),
                roc_auc_score(y_flat, score.ravel())],
    "PR-AUC": [average_precision_score(y_flat, err_only), average_precision_score(y_flat, ad_only),
               average_precision_score(y_flat, score.ravel())],
}, index=["재구성 오차만", "−AssDis만", "둘을 결합"]).round(3)
print(parts.to_string())
print(f"\n무작위 기준 PR-AUC: {y_flat.mean():.4f}")
print("어느 하나만으로는 부족하고, 함께 볼 때 가장 좋아집니다 — 결합이 설계 의도대로 작동했습니다.")

In [ ]:
# 임계값 설정 — 목표 오탐률을 먼저 정하고 분위수로 자른다
"""따라하기"""


In [ ]:
# 임계값 스윕 — 다이얼을 돌리면 무엇이 시소를 타는가
from sklearn.metrics import precision_score, recall_score, f1_score

rates = np.linspace(0.002, 0.06, 30)
rows = []
for r in rates:
    thr = np.quantile(s_flat, 1 - r)
    p = (s_flat >= thr).astype(int)
    rows.append((r, precision_score(y_flat, p, zero_division=0),
                 recall_score(y_flat, p, zero_division=0),
                 f1_score(y_flat, p, zero_division=0)))
sw = np.array(rows)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(sw[:, 0] * 100, sw[:, 1], marker="o", ms=3, label="precision")
ax.plot(sw[:, 0] * 100, sw[:, 2], marker="s", ms=3, label="recall")
ax.plot(sw[:, 0] * 100, sw[:, 3], marker="^", ms=3, label="F1")
ax.axvline(TARGET_ALARM_RATE * 100, ls="--", color="gray")
ax.set_xlabel("alarm rate (%)"); ax.set_title("Threshold sweep — the alarm sensitivity dial")
ax.legend(); plt.tight_layout(); plt.show()
print("알람을 늘리면 잡는 건 늘지만(recall↑) 헛알람도 늘어납니다(precision↓). 공짜 점심은 없습니다.")

In [ ]:
# Accuracy의 무의미함 실증 + 제대로 된 지표 (§7 불균형 평가)
"""따라하기"""


In [ ]:
# ROC vs PR — 불균형에서 어느 곡선을 봐야 하는가
prec, rec, _ = precision_recall_curve(y_flat, s_flat)
fpr, tpr, _ = roc_curve(y_flat, s_flat)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].plot(fpr, tpr); axes[0].plot([0, 1], [0, 1], "r--", lw=0.8)
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].set_title(f"ROC (AUC {auc(fpr, tpr):.3f}) — looks fine")
axes[1].plot(rec, prec)
axes[1].axhline(y_flat.mean(), color="r", ls="--", lw=0.8)
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title(f"PR (AP {average_precision_score(y_flat, s_flat):.3f}) — honest")
plt.tight_layout(); plt.show()
print("ROC의 FPR 분모는 다수인 정상 시점입니다. 오탐 몇 건은 분모에 묻혀 곡선이 좋아 보입니다.")
print("PR은 '알람 중 진짜 비율'을 직접 보여 줍니다 — 현장이 체감하는 숫자입니다.")

---
## 8. [핵심] Attention 히트맵 XAI — 원인 센서 특정

> Attention 히트맵을 읽는 일은 *알람이 울린 순간, 어느 계기판을 보고 있었는지 되짚기* 입니다.

여기가 이 노트북의 하이라이트이자 **난제 ① 해석 가능성의 회수 지점**입니다.
두 단계로 답합니다.

1. **언제·왜 이상인가** — Attention 행렬(시점 × 시점)을 정상 구간과 나란히 비교
2. **어느 센서 때문인가** — 센서별 기여도를 뽑아 Root Cause 특정

In [ ]:
# ① 정상 윈도우 vs 이상 윈도우의 Attention 지도 비교
"""따라하기"""


In [ ]:
# AssDis가 정상과 이상을 실제로 갈라놓았는지 분포로 확인
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.hist(assdis_te.ravel()[y_flat == 0], bins=60, alpha=0.6, density=True, label="normal")
ax.hist(assdis_te.ravel()[y_flat == 1], bins=60, alpha=0.6, density=True, label="anomaly")
ax.set_xlabel("Association Discrepancy"); ax.set_title("Min-Max training pushed the two apart")
ax.legend(); plt.tight_layout(); plt.show()
print("이상 시점이 왼쪽(작은 값)에 몰려 있으면 Min-Max 학습이 의도대로 작동한 것입니다.")

In [ ]:
# ② 센서별 기여도 추출 — 어느 계기판이 문제였는가
"""따라하기"""


In [ ]:
# 이상 구간의 센서별 기여도 히트맵 — 가로축 시간, 세로축 센서
w = w_anom
fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True,
                         gridspec_kw={"height_ratios": [1, 2]})
axes[0].plot(score[w], lw=1.2, color="tab:red")
axes[0].axhline(threshold, ls="--", color="gray")
axes[0].set_ylabel("score")
axes[0].set_title(f"Window #{w}: anomaly score and per-sensor contribution")
im = axes[1].imshow(sensor_err[w].T, aspect="auto", cmap="magma")
axes[1].set_yticks(range(C), SENS, fontsize=8)
axes[1].set_xlabel("time step")
plt.colorbar(im, ax=axes[1], fraction=0.03)
plt.tight_layout(); plt.show()
print("점수가 솟은 시각의 세로줄에서 가장 밝은 칸 — 그 센서가 Root Cause 후보입니다.")
print("현장 엔지니어에게 건네는 것은 '이상입니다'가 아니라 이 그림이어야 합니다.")

> **현장 노트**
> Attention 히트맵을 현장 엔지니어에게 처음 보여 줬을 때 반응은 "그래서 뭐요?" 였습니다.
> 히트맵 자체는 엔지니어의 언어가 아니기 때문입니다.
> 그런데 같은 결과를 **"3번 배럴 온도가 이상 점수의 62%를 차지합니다"** 라는 한 줄로 바꿔
> 전달하자 대화가 달라졌습니다. "그 존 히터 작년에 갈았는데" 하고 곧바로 가설이 나왔습니다.
> XAI의 가치는 그림의 화려함이 아니라 **엔지니어가 자기 경험과 대조할 수 있는 형태**로
> 번역되는 데 있습니다. 모델 출력의 마지막 한 단계는 언제나 번역입니다.

---
## 9. 운영 관점 — 알람 피로와 비용 비대칭

모델을 현장에 걸면 성능표에 없던 문제들이 시작됩니다.

- **알람 피로(alarm fatigue)**: 헛알람이 반복되면 진짜 알람도 무시당합니다. 모델이 아니라 신뢰의 문제입니다.
- **탐지 지연**: 이상 발생부터 알람까지의 시차. 라인이 반응할 시간이 남아 있어야 의미가 있습니다.
- **비용 비대칭**: 오탐 1건 = 라인 정지 비용. 미검출 1건 = 클레임 + 리콜.

In [ ]:
# 임계값을 비용으로 고르기 — 성능이 아니라 손익이 정한다
COST_FP = 200      # 오탐 1건: 불필요한 점검·정지 (만원)
COST_FN = 3000     # 미검출 1건: 클레임·재작업 (만원)

rows = []
for r in [0.005, 0.012, 0.03, 0.06]:
    thr = np.quantile(s_flat, 1 - r)
    p = (s_flat >= thr).astype(int)
    fp = int(((p == 1) & (y_flat == 0)).sum())
    fn = int(((p == 0) & (y_flat == 1)).sum())
    rows.append({"alarm rate": f"{100 * r:.1f}%", "FP": fp, "FN": fn,
                 "recall": round(recall_score(y_flat, p, zero_division=0), 3),
                 "cost(만원)": fp * COST_FP + fn * COST_FN})
cost_tbl = pd.DataFrame(rows)
print(cost_tbl.to_string(index=False))
best_row = cost_tbl.loc[cost_tbl["cost(만원)"].idxmin(), "alarm rate"]
print(f"\n이 비용 구조에서 최소 비용 지점: 알람률 {best_row}")
print("비용 비율이 바뀌면 정답도 바뀝니다 — 그래서 임계값은 현장과 함께 정하는 값입니다.")

In [ ]:
# ★ 평가 단위를 바꾸면 같은 예측의 점수가 달라진다 — 세 가지 프로토콜 비교
def point_adjust(y, pred):
    """이상 구간 중 한 시점이라도 잡으면 구간 전체를 맞힌 것으로 친다 (논문 관행)."""
    p = pred.copy()
    e = np.diff(np.concatenate([[0], y, [0]]))
    for a, b in zip(np.where(e == 1)[0], np.where(e == -1)[0]):
        if pred[a:b].any():
            p[a:b] = 1
    return p

edges = np.diff(np.concatenate([[0], y_flat, [0]]))
starts, ends = np.where(edges == 1)[0], np.where(edges == -1)[0]

rows = []
for r in [0.005, TARGET_ALARM_RATE, 0.02, 0.05]:
    thr = np.quantile(s_flat, 1 - r)
    p = (s_flat >= thr).astype(int)
    hit = sum(1 for a, b in zip(starts, ends) if p[a:b].any())
    rows.append({"알람률": f"{100 * r:.1f}%",
                 "① 시점 F1": round(f1_score(y_flat, p, zero_division=0), 3),
                 "② point-adjust F1": round(f1_score(y_flat, point_adjust(y_flat, p),
                                                    zero_division=0), 3),
                 "③ 구간 재현율": f"{hit}/{len(starts)}"})
proto = pd.DataFrame(rows)
print(proto.to_string(index=False))

base = proto.loc[proto["알람률"] == f"{100 * TARGET_ALARM_RATE:.1f}%"].iloc[0]
print(f"\n알람률 {base['알람률']}에서 — 시점 F1 {base['① 시점 F1']} → "
      f"point-adjust F1 {base['② point-adjust F1']} "
      f"({base['② point-adjust F1'] / max(base['① 시점 F1'], 1e-9):.1f}배)")

lens = ends - starts
print(f"\n이상 구간 {len(starts)}개 | 1스텝짜리 {int((lens == 1).sum())}개, "
      f"2스텝 이상 {int((lens > 1).sum())}개 (최대 {lens.max()}스텝)")

# 탐지 지연 — 구간이 시작되고 몇 스텝 만에 알람이 울렸는가
delays = [int(np.where(s_flat[a:b] >= threshold)[0][0])
          for a, b in zip(starts, ends) if (s_flat[a:b] >= threshold).any()]
if delays:
    print(f"탐지 지연 — 평균 {np.mean(delays):.1f}스텝 / 최대 {max(delays)}스텝 "
          f"(알람률 {100 * TARGET_ALARM_RATE:.1f}% 기준)")
print("지연이 설비 반응 시간보다 길면, 아무리 정확해도 현장에서는 쓸모가 없습니다.")

### 같은 예측인데 점수가 세 배 차이 납니다

| 프로토콜 | 무엇을 재는가 | 성격 |
|---|---|---|
| **① 시점 단위** | 이상인 시점을 정확히 그 시점으로 맞혔는가 | 가장 엄격. 우리가 지금까지 쓴 방식 |
| **② point-adjust** | 구간 중 한 시점만 맞히면 구간 전체를 맞힌 것으로 | **논문 벤치마크의 관행.** 점수가 크게 뜬다 |
| **③ 구간 재현율** | 이상 구간을 놓치지 않았는가 | 현장이 실제로 묻는 질문 |

시점 F1이 낮은 이유는 모델이 나빠서가 아닙니다. 이 데이터의 이상 구간 대부분이
**1스텝짜리 Point 이상**이라, 그 한 시점을 정확히 집어내는 것이 어렵기 때문입니다.
같은 예측을 구간 단위로 보면 대부분 놓치지 않고 잡아냅니다.

> **현장 노트**
> SMAP·SWaT 같은 공개 벤치마크 논문에서 F1 0.9를 봤다면, 대개 ②로 잰 값입니다.
> 무작위 점수로도 F1이 0.9 가까이 나온다는 반박 논문이 있을 만큼 부풀려지는 방식입니다.
> 그래서 저는 남의 성능 보고를 볼 때 **"시점 단위입니까, 구간 단위입니까"** 를 반드시 묻습니다.
> 그리고 우리 프로젝트에서는 **평가 단위를 현장과 먼저 합의**합니다 —
> "몇 초 안에만 알려 주면 됩니까"에 답이 나오면 그게 곧 평가 단위입니다.
> 이 합의 없이 만든 성능 숫자는 나중에 반드시 다시 계산하게 됩니다.

---
## 10. AE vs Anomaly Transformer

세 축으로 정리합니다 — **탐지 성능 / 해석 가능성 / 계산 비용.**

> **비교의 첫 번째 규칙: 예산을 맞춰라.**
> 파라미터가 십수 배 큰 모델을 세 배 오래 학습시킨 뒤 "이 모델이 낫다"고 말하는 것은
> 비교가 아닙니다. 아래 표에는 **AT와 같은 규모로 맞춘 AE**를 먼저 놓고,
> **훨씬 크게 키운 AE**(AT 대비 약 17배)를 참고로 함께 실었습니다.

### 표를 보기 전에 — 지표를 어떻게 읽는가

| 지표 | 무엇을 재는가 | 무작위일 때 | 이상탐지에서의 의미 |
|---|---|---|---|
| **ROC-AUC** | 이상 시점이 정상 시점보다 높은 점수를 받을 확률 | 항상 **0.5** | 순위를 잘 매기는가. 불균형에 둔감해 실제보다 후하게 나온다 |
| **PR-AUC** | 알람을 울렸을 때 그것이 진짜일 비율(정밀도)의 평균 | **이상 비율**(여기서는 약 0.014) | 현장이 체감하는 값. 기저율이 낮으면 절대값도 낮게 나온다 |
| **원인 적중률** | 지목한 원인 센서가 실제 원인과 일치한 비율 | **1/센서 수** = 0.125 | 알람에 근거가 따라오는가 |

**PR-AUC를 읽을 때 반드시 기저율과 함께 보십시오.**
이상 비율이 1.4%인 데이터에서 무작위 추측의 PR-AUC는 0.014입니다.
따라서 PR-AUC 0.5는 "절반밖에 못 맞혔다"가 아니라 **무작위 대비 약 35배**라는 뜻입니다.
같은 이유로 **서로 다른 데이터셋의 PR-AUC를 직접 비교하면 안 됩니다** — 기저율이 다르기 때문입니다.

그래서 아래 표의 첫 줄에 **모델 없이 단순 통계만 쓴 베이스라인**을 넣었습니다.
*베이스라인은 신입 작업자의 감(勘)입니다 — 이걸 못 이기면 모델은 필요 없습니다.*

In [ ]:
# 최종 비교표 — 베이스라인부터 순서대로
"""따라하기"""


**표를 읽는 법 — 네 줄이 하나로 이어집니다**

1. **단순 통계도 만만치 않습니다.** Point 이상(뾰족한 스파이크)은 모델 없이도 잡힙니다.
   여기를 못 넘으면 모델을 도입할 이유가 없습니다.
2. **같은 예산이면 구조가 이깁니다.** 파라미터가 비슷할 때 Anomaly Transformer가
   AE를 크게 앞섭니다. "닮은 과거가 있는가"라는 질문이 재구성 오차보다 강한 신호이기 때문입니다.
   특히 값이 정상 범위인 **Contextual 이상**에서 격차가 벌어집니다 — 단순 통계로는 손도 못 대는 유형입니다.
3. **AE도 크게 키우면 따라옵니다.** 다만 파라미터 17배와 학습 시간을 지불해야 하고,
   §3에서 봤듯 크게 키울수록 Over-generalization 위험도 함께 커집니다.
4. **그렇게 해도 AE는 "왜"를 말하지 못합니다.** 원인 센서 지목은 AT만 해냈고,
   이것이 오늘 회수한 난제 ① 그 자체입니다.

**그래도 AE를 고를 때가 있습니다.** 이상이 명백한 형태로 나타나는 공정
(오늘 앞부분의 펌프 정지처럼)에서는 AE로 충분하고, 가볍고 구현 부담이 적다는 것이
실제 이점이 됩니다. **알고리즘의 우열은 이상의 성격이 정합니다** —
Point·Contextual 이상이 흩어져 있으면 Association 기반이 유리하고,
구간 전체가 다른 상태로 바뀌면 재구성 오차만으로도 충분합니다.

> ※ 학습 데이터가 적고 이상이 희소해 숫자는 실행마다 흔들립니다.
> 한 번의 실행으로 순위를 단정하지 말고, 우리 데이터에서 여러 번 재보고 판단하십시오.

> **현장 노트**
> 제가 본 이상탐지 PoC의 절반은 "어떤 모델을 쓸까"에서 시작해 실패했습니다.
> 순서가 반대입니다. **우리 라인의 불량이 Point인가 Contextual인가 Collective인가**,
> 라벨은 언제 붙는가, 오탐과 미검출 중 무엇이 더 비싼가 — 이 세 질문에 답한 뒤에
> 모델을 고르면 후보는 대개 한두 개로 줄어듭니다.

---
## Self-check

### Q1. AutoEncoder의 Over-generalization이란 무엇이며, 어떤 조건에서 심해집니까?

<details>
<summary>정답 보기</summary>

- AE는 "입력을 잘 복원하라"만 배웠을 뿐 "이상은 복원하지 말라"는 배운 적이 없습니다.
- 학습이 길어질수록 처음 보는 이상 파형까지 복원해, 정상과 이상의 오차 차이가 줄어듭니다.
- 실습의 펌프 테스트베드에서도 재구성 손실은 계속 내려가는데 탐지 ROC-AUC는 정점을 찍고 꺾였습니다.
- 다만 **항상 일어나지는 않습니다.** 이상이 정상 분포에서 멀수록(설비 정지처럼) AE는 잘 버팁니다.
  이상이 정상과 가까울수록 심해지므로, 우리 공정의 이상이 어느 쪽인지가 판단 기준입니다.
- 실무 대응은 하나입니다 — **손실이 아니라 탐지 지표로 조기 종료**를 판단합니다.

</details>

---

### Q2. Association Discrepancy가 이상 시점에서 작아지는 이유를 설명하십시오.

<details>
<summary>정답 보기</summary>

- 정상 시점은 주기적으로 닮은 과거가 있어 Attention(Series)이 멀리까지 퍼집니다. 인접 집중을 가정한 Prior와 크게 달라 AssDis가 커집니다.
- 이상 시점은 과거에 닮은 순간이 없어 바로 옆 시점하고만 대화합니다. 그 결과 Series가 Prior와 닮아 AssDis가 작아집니다.
- Min-Max 학습은 Prior를 현실에 맞추면서(minimize) Series는 Prior에서 밀어내(maximize) 두 집단의 간격을 넓힙니다.

</details>

---

### Q3. 이상 비율 1%인 데이터에서 ROC 곡선보다 PR 곡선을 보라고 하는 이유는 무엇입니까?

<details>
<summary>정답 보기</summary>

- ROC의 FPR은 분모가 다수인 정상 시점이라, 오탐이 수십 건 늘어도 곡선이 거의 움직이지 않습니다.
- PR의 Precision은 "울린 알람 중 진짜 비율"을 직접 보여 주며, 이것이 현장이 체감하는 숫자입니다.
- 무작위 기준선도 다릅니다 — ROC는 항상 0.5지만 PR은 이상 비율(여기서는 약 0.01)이라 비교 기준이 정직합니다.

</details>

---

### Q4. [현장 판단] 모델이 "지금 이상입니다"라고 알렸습니다. 이 출력을 그대로 현장에 전달하면 안 되는 이유는 무엇이며, 무엇을 덧붙여야 합니까?

<details>
<summary>정답 보기</summary>

- 근거 없는 알람은 검증할 방법이 없어 무시되고, 반복되면 알람 피로로 이어져 진짜 알람까지 묻힙니다.
- **어느 센서가 얼마나 기여했는지**(원인 센서와 비중), 언제부터 시작됐는지(탐지 지연), 과거 유사 사례를 함께 전달해야 합니다.
- 히트맵 그대로가 아니라 "3번 배럴 온도가 이상 점수의 62%"처럼 엔지니어의 언어로 번역해야 대화가 시작됩니다.
- 덧붙여, 그 원인 지목이 실제로 맞는지 정답이 있는 구간에서 미리 채점해 두어야 신뢰를 얻을 수 있습니다.

</details>


---
## 다음 노트북 예고 — NB04. 표현학습(SSL)과 도메인 적응

난제 ①은 회수했습니다. 그런데 오늘 실습에는 조용한 전제가 하나 깔려 있었습니다.
**"학습에 쓸 정상 데이터가 충분히 있다"** 는 것입니다.

- 지난주 증설한 신규 라인에는 그 정상 데이터조차 없습니다 → **난제 ② Cold-Start**
- 어제까지 맞던 모델이 부품 교체 한 번에 무너집니다 → **난제 ③ Concept Drift**

다음 노트북에서는 **라벨 없이 표현을 배우는 자기지도학습(SSL)** 으로 ②를,
**도메인 적응(MMD)** 으로 ③을 회수합니다.
NB02에서 다룬 엔진 데이터의 FD001 → FD003 전이가 무대이고,
*같은 엔진인데 운전 조건이 달라 모델이 무너지는* 장면을 실데이터로 확인합니다.

---
## 📎 부록 — `"""직접구현"""` 정답 코드

먼저 스스로 채워 본 뒤에 펼쳐 보시기 바랍니다.

<details>
<summary>Association Discrepancy — 두 분포의 대칭 KL 거리</summary>

```python
# Association Discrepancy — 두 분포의 대칭 KL 거리
"""직접구현"""
def assoc_discrepancy(P, S, eps=1e-8):
    """P, S: (B, h, L, L) → 시점별 불일치도 (B, L)"""
    kl_ps = (P * (torch.log(P + eps) - torch.log(S + eps))).sum(-1)   # KL(P||S)
    kl_sp = (S * (torch.log(S + eps) - torch.log(P + eps))).sum(-1)   # KL(S||P)
    return (kl_ps + kl_sp).mean(1)                                    # head 평균

# 검증: 옆만 보는 Attention은 prior와 닮아 AssDis가 작아야 한다
P_demo = make_prior(40, 3.0)[None, None]
S_near = make_prior(40, 3.0)[None, None]                 # 이상 시점처럼 인접만
S_far = make_prior(40, 15.0)[None, None]                 # 정상 시점처럼 멀리까지
print(f"이상처럼 옆만 볼 때 AssDis: {assoc_discrepancy(P_demo, S_near).mean():.4f}")
print(f"정상처럼 멀리 볼 때 AssDis: {assoc_discrepancy(P_demo, S_far).mean():.4f}")
print("→ 이상 시점의 AssDis가 더 작습니다. 이 성질이 탐지의 근거가 됩니다.")
```

</details>

<details>
<summary>이상 점수 산출 — AssDis와 재구성 오차를 전체 기준으로 표준화해 결합</summary>

```python
# 이상 점수 산출 — AssDis와 재구성 오차를 전체 기준으로 표준화해 결합
"""직접구현"""
def anomaly_score(model, X, batch=64):
    """반환: score (N, L), sensor_err (N, L, C), assdis (N, L), series_attn (N, L, L)"""
    model.eval(); Es, Ad, At = [], [], []
    with torch.no_grad():
        for i in range(0, len(X), batch):
            xb = torch.tensor(X[i:i + batch]).to(DEVICE)
            rec, Ps, Ss = model(xb)
            Es.append(((rec - xb) ** 2).cpu().numpy())                # (B, L, C) 센서별 오차
            ad = torch.stack([assoc_discrepancy(P, S) for P, S in zip(Ps, Ss)]).mean(0)
            Ad.append(ad.cpu().numpy())
            At.append(Ss[-1].mean(1).cpu().numpy())                   # head 평균 Attention
    sensor_err = np.concatenate(Es); assdis = np.concatenate(Ad); attn = np.concatenate(At)
    err = sensor_err.mean(-1)                                          # (N, L)
    z = lambda v: (v - v.mean()) / (v.std() + 1e-9)                    # 전체 구간 기준 표준화
    score = z(err) + z(-assdis)                                        # 두 신호를 같은 저울에 올려 합산
    return score, sensor_err, assdis, attn

score, sensor_err, assdis_te, attn = anomaly_score(model, X_te)
print(f"score {score.shape} | 센서별 오차 {sensor_err.shape} | attention {attn.shape}")
print(f"정상 시점 평균 AssDis {assdis_te.ravel()[y_flat == 0].mean():.4f} | "
      f"이상 시점 평균 AssDis {assdis_te.ravel()[y_flat == 1].mean():.4f}")
```

</details>
